# Topic 4: Spill & OOM Diagnosis

> **Phase 4 → Week 6 → Topic 4**

---

## The Memory Pressure Escalation

```
Memory usage increases →

Level 1: NORMAL
  Execution memory used for shuffle/sort/agg
  Storage memory used for cache
  → Fast, everything in RAM

Level 2: CACHE EVICTION
  Execution needs more memory
  → LRU cached partitions evicted to disk
  → Next access of evicted partition = recompute
  → Slower, but recoverable

Level 3: SPILL TO DISK
  Sort or aggregation buffer full, no more memory available
  → Partial results written to local disk as a "spill file"
  → Later merged from disk (spill read + merge)
  → Significantly slower (disk I/O) but task completes

Level 4: OUT OF MEMORY (OOM)
  • Java heap exhausted (no spill path for this operation)
  • Broadcast table too large (no spill for broadcast)
  • Python worker OOM (UDF data too large)
  → Task fails → executor may die → job fails or retries
```

---

## Common OOM Causes

| Cause | Error message | Fix |
|-------|--------------|-----|
| Broadcast table too large | `SparkException: Size exceeds limit` | Reduce `autoBroadcastJoinThreshold` or use SMJ |
| collect() too much data | `OutOfMemoryError` on driver | Filter before collect, use `take()` or write to disk |
| Python UDF data too large | `MemoryError` in Python | Increase `executor.memoryOverhead` |
| Large sort/shuffle | `OutOfMemoryError` executor | Increase executor memory or partitions |
| Cache fills heap | `OutOfMemoryError` executor | Unpersist unused DFs, reduce `storageFraction` |
| Skewed data | One task OOM | Fix skew (salting, AQE) |

---

## Interview Cheat Sheet

**Q: What is spill in Spark and when does it happen?**
> Spill occurs when an operation (sort, aggregation, shuffle) runs out of in-memory execution space. Partial results are written to local disk as temporary spill files, then merged back later. It's visible in Spark UI as "Spill (Memory)" and "Spill (Disk)" columns on stage detail. Spill causes slow tasks but doesn't fail them.

**Q: How do you diagnose an OOM in Spark?**
> Check where it occurred: executor OOM (in task logs, look for `java.lang.OutOfMemoryError`) vs driver OOM. For executor: check stage metrics for spill, skew (huge input size for one task), and broadcast join sizes. For driver: usually `collect()` of too much data. Steps: (1) check UI for task size outliers, (2) check for broadcast hints on large tables, (3) check if skew is causing one partition to explode.

**Q: What's the difference between executor OOM and driver OOM?**
> Executor OOM: a task running on an executor exhausts JVM heap — the executor dies, tasks are rescheduled to other executors (up to `spark.task.maxFailures` retries). Driver OOM: the driver process runs out of memory — usually from `collect()` pulling too much data, or creating a huge broadcast variable. Driver OOM is fatal — the entire application fails.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
import time

spark = SparkSession.builder \
    .appName("Week6 - Spill and OOM") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:04:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


## Part 1: Spill — Detecting and Measuring

In [2]:
# To observe spill, we need to reduce memory available to Spark
# In local mode, we can force spill by setting very low memory limits
spark.conf.set("spark.sql.shuffle.partitions", "2")

# Large aggregation that should trigger spill if executor memory is constrained
# Use wide rows to consume more memory per record
df_large = spark.range(2_000_000) \
    .withColumn("key",   (F.col("id") % 10).cast("string")) \
    .withColumn("data1", F.repeat(F.lit("x"), 50)) \
    .withColumn("data2", F.rand() * 1000) \
    .withColumn("data3", F.col("id").cast("double"))

print("Running aggregation — check Spark UI → Stages for Spill columns")
result = df_large.groupBy("key").agg(
    F.sum("data2").alias("total"),
    F.avg("data3").alias("avg_id"),
    F.count("*").alias("cnt")
)
t0 = time.time()
result.show()
print(f"Completed in {time.time()-t0:.2f}s")
print()
print("Look in Spark UI → Stages → click stage → Task Metrics:")
print("  'Spill (Memory)' = bytes that were in memory before spill")
print("  'Spill (Disk)'   = bytes written to disk during spill")
spark.conf.set("spark.sql.shuffle.partitions", "4")

Running aggregation — check Spark UI → Stages for Spill columns


+---+--------------------+---------+------+
|key|               total|   avg_id|   cnt|
+---+--------------------+---------+------+
|  1|1.0009787973906805E8| 999996.0|200000|
|  3| 9.996649452493252E7| 999998.0|200000|
|  4| 9.987643205445492E7| 999999.0|200000|
|  8| 9.988090967562836E7|1000003.0|200000|
|  0|1.0002021398746805E8| 999995.0|200000|
|  2| 9.988263472848976E7| 999997.0|200000|
|  5|1.0002886021324381E8|1000000.0|200000|
|  6| 9.994338959023933E7|1000001.0|200000|
|  7|1.0014228821348757E8|1000002.0|200000|
|  9|1.0001815290829462E8|1000004.0|200000|
+---+--------------------+---------+------+

Completed in 4.29s

Look in Spark UI → Stages → click stage → Task Metrics:
  'Spill (Memory)' = bytes that were in memory before spill
  'Spill (Disk)'   = bytes written to disk during spill


## Part 2: Diagnosing Spill Sources

In [3]:
print("""
Operations That Spill (and their fixes):
════════════════════════════════════════════════════════════════
Operation          Spills when...            Fix
────────────────   ───────────────────────   ─────────────────────────
groupBy + agg      Hash table > exec memory  • More partitions (smaller per-partition data)
                                             • Increase executor memory
                                             • Enable AQE (auto-coalesce)

orderBy / sort     Sort buffer > exec memory  • More partitions
                                             • Avoid full sort; use sortWithinPartitions
                                               if global order not required

SortMergeJoin      Sort buffers > exec memory • Broadcast the smaller side
                                             • Increase executor memory
                                             • More shuffle partitions

Window function    Window frame too large     • Reduce window frame size
                                             • Filter before window

Distinct           Hash table for dedup       • Increase partitions
════════════════════════════════════════════════════════════════
Golden rule: SPILL is caused by too much data per partition.
Fix: MORE PARTITIONS (smaller data per partition)
""")


Operations That Spill (and their fixes):
════════════════════════════════════════════════════════════════
Operation          Spills when...            Fix
────────────────   ───────────────────────   ─────────────────────────
groupBy + agg      Hash table > exec memory  • More partitions (smaller per-partition data)
                                             • Increase executor memory
                                             • Enable AQE (auto-coalesce)

orderBy / sort     Sort buffer > exec memory  • More partitions
                                             • Avoid full sort; use sortWithinPartitions
                                               if global order not required

SortMergeJoin      Sort buffers > exec memory • Broadcast the smaller side
                                             • Increase executor memory
                                             • More shuffle partitions

Window function    Window frame too large     • Reduce window frame size
            

In [4]:
# Fixing spill with more partitions
import time

def run_groupby(partitions: int):
    spark.conf.set("spark.sql.shuffle.partitions", str(partitions))
    t0 = time.time()
    df_large.groupBy("key").agg(F.sum("data2")).count()
    elapsed = time.time() - t0
    print(f"  shuffle.partitions={partitions:>4d} → {elapsed:.3f}s")

print("Effect of partition count on spill-prone aggregation:")
for n in [1, 2, 4, 8, 16]:
    run_groupby(n)

print()
print("More partitions = smaller hash tables per partition = less spill")
spark.conf.set("spark.sql.shuffle.partitions", "4")

Effect of partition count on spill-prone aggregation:


  shuffle.partitions=   1 → 0.918s


  shuffle.partitions=   2 → 1.011s


  shuffle.partitions=   4 → 0.716s


  shuffle.partitions=   8 → 0.361s


  shuffle.partitions=  16 → 0.365s

More partitions = smaller hash tables per partition = less spill


## Part 3: OOM Patterns and Fixes

In [5]:
# PATTERN 1: Driver OOM from collect()
# The collect() anti-pattern — pulling ALL data to driver

big_result = spark.range(1_000_000).withColumn("v", F.rand())

# BAD: collect() pulls 1M rows to driver RAM
# big_result.collect()  # DON'T RUN — could OOM the driver!

print("DON'T: df.collect() on a large DataFrame")
print()

# GOOD alternatives to collect()
print("GOOD alternatives:")

# 1. take(N) — only pull first N rows
sample = big_result.take(10)
print(f"  take(10): returns {len(sample)} rows to driver")

# 2. show() — prints to stdout, doesn't return to Python
big_result.filter(F.col("v") > 0.999).show(5)

# 3. Write to disk — never bring to driver at all
big_result.write.mode("overwrite").parquet("/tmp/output_no_collect")
print("  write.parquet(): writes to storage, driver never holds the data")

# 4. Aggregate then collect (small result)
agg_result = big_result.agg(F.sum("v"), F.avg("v"), F.count("*")).collect()
print(f"  aggregate then collect: {len(agg_result)} row(s) to driver (safe)")

DON'T: df.collect() on a large DataFrame

GOOD alternatives:


  take(10): returns 10 rows to driver


+----+------------------+
|  id|                 v|
+----+------------------+
| 278|0.9990586370055312|
| 396|0.9990258741565115|
|1033|0.9999874414443348|
|3324|0.9991035128135203|
|3562|0.9999227899161752|
+----+------------------+
only showing top 5 rows



  write.parquet(): writes to storage, driver never holds the data


  aggregate then collect: 1 row(s) to driver (safe)


In [6]:
# PATTERN 2: Broadcast OOM
# Broadcasting a table that's too large for executor memory

print("""
Broadcast OOM Fix:
─────────────────────────────────────────────────────────────
Error: SparkException: Cannot broadcast... size exceeds threshold
       or: java.lang.OutOfMemoryError during BroadcastHashJoin

Cause: broadcast() hint used on a table that doesn't fit in
       executor memory after deserialization

Fix 1: Remove broadcast hint → use SortMergeJoin (spill-safe)
  orders.join(products, ...)           # no broadcast hint

Fix 2: Reduce the broadcast table (filter before broadcast)
  orders.join(
    broadcast(products.filter(col('category') == 'Electronics')),
    ...)

Fix 3: Lower autoBroadcastJoinThreshold to stop accidental broadcasts
  spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '5m')

Fix 4: Increase executor memory (last resort)
─────────────────────────────────────────────────────────────
""")


Broadcast OOM Fix:
─────────────────────────────────────────────────────────────
Error: SparkException: Cannot broadcast... size exceeds threshold
       or: java.lang.OutOfMemoryError during BroadcastHashJoin

Cause: broadcast() hint used on a table that doesn't fit in
       executor memory after deserialization

Fix 1: Remove broadcast hint → use SortMergeJoin (spill-safe)
  orders.join(products, ...)           # no broadcast hint

Fix 2: Reduce the broadcast table (filter before broadcast)
  orders.join(
    broadcast(products.filter(col('category') == 'Electronics')),
    ...)

Fix 3: Lower autoBroadcastJoinThreshold to stop accidental broadcasts
  spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '5m')

Fix 4: Increase executor memory (last resort)
─────────────────────────────────────────────────────────────



In [7]:
# PATTERN 3: Skewed partition OOM
# One partition gets all the data → one task OOMs while others are fine

skewed = spark.createDataFrame(
    [("A", i) for i in range(50_000)] +
    [("B", i) for i in range(100)    ] +
    [("C", i) for i in range(100)    ],
    ["key", "val"]
)

print("Partition size distribution (skewed data):")
skewed.groupBy("key").count().show()

print("""
When joined or grouped, key='A' rows all go to ONE partition.
That one task gets 99% of data → huge memory pressure → OOM or spill.

Symptoms:
  • Spark UI shows 1 task taking 100x longer than others
  • That task has massive 'Input Size' (or 'Shuffle Read Size')
  • Stage doesn't finish until that one task completes

Fixes:
  1. Salting (covered in Week 7 skew notebook)
  2. AQE skew join (spark.sql.adaptive.skewJoin.enabled=true)
  3. Filter out the skewed key, process separately, union
""")

Partition size distribution (skewed data):


+---+-----+
|key|count|
+---+-----+
|  A|50000|
|  B|  100|
|  C|  100|
+---+-----+


When joined or grouped, key='A' rows all go to ONE partition.
That one task gets 99% of data → huge memory pressure → OOM or spill.

Symptoms:
  • Spark UI shows 1 task taking 100x longer than others
  • That task has massive 'Input Size' (or 'Shuffle Read Size')
  • Stage doesn't finish until that one task completes

Fixes:
  1. Salting (covered in Week 7 skew notebook)
  2. AQE skew join (spark.sql.adaptive.skewJoin.enabled=true)
  3. Filter out the skewed key, process separately, union



## Part 4: OOM Diagnosis Workflow

In [8]:
print("""
OOM Diagnosis Workflow:
═══════════════════════════════════════════════════════════════
Step 1: Identify WHERE the OOM happened
  □ Driver OOM?   → look for error in your notebook/application logs
  □ Executor OOM? → Spark UI → Executors tab → shows executor lost
                    Spark UI → Stages → failed task details

Step 2: Driver OOM path
  □ Are you using collect() on a large result? → use write() instead
  □ Are you creating large broadcast variables? → filter first
  □ Fix: spark.driver.memory = 4g or 8g

Step 3: Executor OOM path
  □ Check Stage detail → is one task much larger? (skew)
     If yes → apply salting or AQE skew join
  □ Check for BroadcastHashJoin in explain() with a large table
     If yes → remove broadcast hint or lower threshold
  □ Check Spill columns in stage detail
     If spill > 0 → increase partitions first
     If spill = 0 and OOM → need more executor memory

Step 4: Tuning order (try cheapest first)
  1. Increase shuffle.partitions (free)
  2. Fix skew with AQE (free, enable if not already)
  3. Remove broadcast hints on large tables (free)
  4. Increase executor.memory (costs more resources)
  5. Add more executors (costs more resources)
═══════════════════════════════════════════════════════════════
""")


OOM Diagnosis Workflow:
═══════════════════════════════════════════════════════════════
Step 1: Identify WHERE the OOM happened
  □ Driver OOM?   → look for error in your notebook/application logs
  □ Executor OOM? → Spark UI → Executors tab → shows executor lost
                    Spark UI → Stages → failed task details

Step 2: Driver OOM path
  □ Are you using collect() on a large result? → use write() instead
  □ Are you creating large broadcast variables? → filter first
  □ Fix: spark.driver.memory = 4g or 8g

Step 3: Executor OOM path
  □ Check Stage detail → is one task much larger? (skew)
     If yes → apply salting or AQE skew join
  □ Check for BroadcastHashJoin in explain() with a large table
     If yes → remove broadcast hint or lower threshold
  □ Check Spill columns in stage detail
     If spill > 0 → increase partitions first
     If spill = 0 and OOM → need more executor memory

Step 4: Tuning order (try cheapest first)
  1. Increase shuffle.partitions (free)
  2. Fi

## Exercises

1. You see `Spill (Disk) = 2.1 GB` in the Spark UI for a sort stage. What does this mean and what are 3 ways to fix it?
2. A job fails with `java.lang.OutOfMemoryError: Java heap space` on the driver. You're not calling `collect()`. What else could cause driver OOM?
3. Write code that demonstrates the dangerous `collect()` pattern and its safe alternative.
4. A join stage shows: Task 1 processes 5MB, Task 2 processes 5MB, Task 3 processes 4.8GB. What is this called and how do you fix it?
5. What is `spark.task.maxFailures` and why should you set it to 4 instead of 1 in production?

In [9]:
# Exercise 3: Dangerous collect() vs safe alternatives
medium_df = spark.range(100_000).withColumn("v", F.rand())

# DANGEROUS (for production scale): 
# rows = medium_df.collect()  # pulls ALL to driver RAM

# SAFE alternatives:
print("Safe alternative 1: take(N)")
rows = medium_df.take(5)
print(f"  Got {len(rows)} rows")

print("Safe alternative 2: aggregate then collect")
agg = medium_df.agg(F.count("*"), F.avg("v")).collect()
print(f"  Got 1 summary row: {agg}")

print("Safe alternative 3: write to storage")
medium_df.write.mode("overwrite").parquet("/tmp/safe_output")
print("  Written to /tmp/safe_output")

print("Safe alternative 4: topN via orderBy + limit")
top5 = medium_df.orderBy(F.col("v").desc()).limit(5).collect()
print(f"  Top 5 values: {[round(r.v, 3) for r in top5]}")

Safe alternative 1: take(N)
  Got 5 rows
Safe alternative 2: aggregate then collect


  Got 1 summary row: [Row(count(1)=100000, avg(v)=0.4987761898929601)]
Safe alternative 3: write to storage


  Written to /tmp/safe_output
Safe alternative 4: topN via orderBy + limit


  Top 5 values: [1.0, 1.0, 1.0, 1.0, 1.0]
